# 第 24 节: RL 面试复习

## 覆盖全部核心面试题

每道题分三个层次回答:
- 30 秒回答 (面试开场)
- 2 分钟回答 (深入追问)
- 深度追问 (展示专业水平)

## Q1: 什么是 MDP?

<details><summary>答案</summary>

**30秒**: MDP (Markov Decision Process) 是 RL 的数学框架, 由 (S, A, P, R, gamma) 五元组定义. 核心假设是 Markov 性质 -- 未来只取决于当前状态.

**2分钟**: MDP 形式化了一个决策问题: 在每个状态 s, 智能体选择动作 a, 以概率 P(s'|s,a) 转移到新状态并收到奖励 R(s,a). 目标是找到最大化期望累积折扣回报的策略. gamma 控制远见 vs 近利.

**追问**: 有限 MDP 的 Bellman 最优方程有唯一不动点 (由 contraction 性质保证). POMDP 是 MDP 的推广, 状态不完全可观测.
</details>

## Q2: MC 和 TD 的区别?

<details><summary>答案</summary>

**30秒**: MC 等 episode 结束后用实际回报更新, TD 每步用 bootstrapping 更新. MC 无偏但高方差, TD 有偏但低方差.

**2分钟**: MC target 是 G_t (实际回报), TD target 是 r + gamma*V(s') (估计+实际). TD 利用了 MDP 结构 (Bellman 方程), 所以样本效率更高. TD(0) 可以看作对 Bellman 方程的随机近似.

**追问**: n-step TD 和 TD(lambda) 提供了 MC 和 TD(0) 之间的平滑过渡. lambda=0 是 TD(0), lambda=1 是 MC.
</details>

## Q3: Policy Gradient Theorem 推导

<details><summary>答案</summary>

**30秒**: grad J = E[grad log pi(a|s) * Q(s,a)]. 用 log-derivative trick 把对轨迹概率的梯度转为对 log 策略的梯度.

**2分钟**: 核心三步: 1) J = E_tau[R(tau)] = integral P(tau|theta)R(tau); 2) grad P = P * grad log P (log-derivative trick); 3) 环境动态的梯度为零 (不依赖 theta). 最终得到 grad J = E[sum_t grad log pi(a_t|s_t) * G_t].

**追问**: Reward-to-go 形式比 trajectory return 有更低方差, 因为 a_t 不影响 t 之前的奖励 (因果性).
</details>

## Q4: On-Policy vs Off-Policy

<details><summary>答案</summary>

**30秒**: On-policy 学习的策略 = 采样的策略 (SARSA, PPO). Off-policy 学习的策略 != 采样的策略 (Q-Learning, DQN).

**2分钟**: On-policy 更稳定但样本效率低 (每次更新后必须用新策略重新采样). Off-policy 可以复用旧数据 (Experience Replay), 但需要重要性采样修正, 可能导致高方差或不稳定.

**追问**: The Deadly Triad: function approximation + bootstrapping + off-policy = potential divergence.
</details>

## Q5: DQN 的三个关键创新?

<details><summary>答案</summary>

1. Experience Replay: 打破数据相关性, 提高数据效率
2. Target Network: 固定 TD target 中的网络, 稳定训练
3. Reward Clipping / Huber Loss: 控制梯度 scale

**追问**: Double DQN 用 online net 选动作, target net 评估, 解决 overestimation bias. Dueling DQN 将 Q 分解为 V(s) + A(s,a).
</details>

## Q6: PPO 的核心公式和 clipping 机制

<details><summary>答案</summary>

**30秒**: L = E[min(r*A, clip(r, 1-e, 1+e)*A)]. 通过 clipping 限制策略更新幅度, 是 TRPO 的简化版.

**2分钟**: r = pi_new/pi_old. 当 A>0 时, r 被限制在 1+e (防止过于激进); 当 A<0 时, r 被限制在 1-e (防止过于保守). 这相当于隐式地约束 KL divergence.

**追问**: PPO clipping 不是严格的 trust region (只限制一个方向), 但在实践中非常有效.
</details>

## Q7: GAE 中 lambda 和 gamma 的作用

伽马 (折扣因子): 控制我们多关心未来奖励。伽马趋于 1 = 远视, 伽马趋于 0 = 短视
Lambda (GAE 参数): 控制 bias-variance trade-off。Lambda 趋于 1 = MC (低偏差高方差), Lambda 趋于 0 = TD(0) (高偏差低方差)

GAE 将 advantage 估计为 TD errors 的指数加权和: A_GAE = sum (gamma*lambda)^l * delta_{t+l}.
</details>

## Q8: 如何设计 Reward?

- 稀疏 vs 稠密: 稀疏奖励 (如围棋的输赢) 难学, 需要额外探索机制
- Reward shaping: 添加辅助奖励, 但要小心不引入意外行为
- Scale: reward scale 影响梯度大小, 推荐归一化到 [-1, 1] 或 [0, 1]
- Potential-based shaping: 能保证最优策略不变
</details>

## Q9: RL 不稳定的原因?

1. 非平稳数据分布: 策略改变导致数据分布改变
2. Bootstrapping 的误差传播: 不准确的估计用于更新其他估计
3. 策略的敏感性: 策略参数小变化导致行为大变化导致数据分布变化
4. Deadly Triad: FA + Bootstrapping + Off-policy = 可能发散
5. 高方差梯度: REINFORCE/MC 的梯度方差大
</details>

## Q10: Terminated vs Truncated

- Terminated: 环境自然终止 (到达目标, 游戏结束)
- Truncated: 因外部限制终止 (达到 max_steps, 时间到)

在 bootstrap 中:
- Terminated -> 不 bootstrap (未来价值为 0)
- Truncated -> 需要 bootstrap (因为环境本来可以继续)

混淆二者会导致:
- 将 truncated 当作 terminated -> 错误地将非零未来价值设为零
- 将 terminated 当作 truncated -> 对不可能继续的状态进行 bootstrap
</details>

---
下一节: 25_final_assessment.ipynb